# [1장 1강] - 실습: 벡터·노름·정규화

In [ ]:
%pip install -q numpy
%pip install -q pandas
%pip install -q matplotlib
%pip install -q scikit-learn
%pip install -q ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

np.set_printoptions(linewidth=200)

In [ ]:
def _fallback(reason):
    """UCI 데이터를 쓸 수 없을 때 대체 데이터셋을 반환합니다."""
    print(f'[경고] {reason}')
    print('[경고] UCI Wine Quality를 불러오지 못해 다른 데이터셋(sklearn wine)으로 대체됩니다.')
    print('[경고] 따라서 아래 출력값은 교안의 예시 값과 다르게 나옵니다. 계산·해석 방법은 동일합니다.')
    from sklearn.datasets import load_wine
    data = load_wine(as_frame=True)
    return data.data, data.target


def load_uci(dataset_id):
    """UCI에서 데이터를 불러오고, 실패 원인(설치/네트워크)을 구분해 안내합니다."""
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError:
        return _fallback('ucimlrepo 패키지가 설치되어 있지 않습니다. 위의 pip 설치 셀을 먼저 실행하세요.')

    try:
        ds = fetch_ucirepo(id=dataset_id)
    except Exception as e:
        return _fallback(f'UCI 서버 접속에 실패했습니다(네트워크·방화벽 확인 필요): {e}')

    X, y = ds.data.features.copy(), ds.data.targets.copy()
    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]  # 컬럼 1개짜리 DataFrame -> Series
    return X, y


def numeric_frame(X):
    """수치형 컬럼만 남기고 결측값을 중앙값으로 채웁니다."""
    Xn = X.select_dtypes(include='number').copy()
    Xn = Xn.replace([np.inf, -np.inf], np.nan)
    return Xn.fillna(Xn.median(numeric_only=True))


X_raw, y_raw = load_uci(186)   # Wine Quality
X = numeric_frame(X_raw)
print('데이터 shape:', X.shape)

# 필수 1 : 품질 측정 기록을 벡터로 다루기
와인 제조사 품질관리팀은 생산 샘플마다 산도·당도·알코올 같은 이화학 값을 여러 개 측정해 기록합니다. 값이 여러 개이므로 샘플 하나를 숫자 하나로는 표현할 수 없고, "측정값 묶음" 단위로 다뤄야 합니다. 먼저 이 기록이 스칼라·벡터·행렬 중 무엇에 해당하는지 정리하고, 샘플 간 비교에 필요한 기본 연산을 익힙니다.

### 문제 1-1 : 스칼라·벡터·행렬 구분하기
1. `X`에서 첫 번째 샘플의 측정값을 `v1`(NumPy 배열)로 추출합니다.
2. `X` 전체 표, `v1`, `v1`의 첫 번째 원소를 각각 출력하고 `shape`(또는 값)을 확인합니다.
3. 세 대상이 각각 행렬·벡터·스칼라 중 무엇인지, 분석에서 어떤 역할을 하는지 한 문장씩 설명합니다.

In [ ]:
# 1. `X`에서 첫 번째 샘플의 측정값을 `v1`(NumPy 배열)로 추출합니다.
print(X.head(1))

v1 = X.iloc[0].to_numpy()
# v1 = X.iloc[0].values

print("====================")
print(v1)
print(type(v1))

In [ ]:
# 2. `X` 전체 표, `v1`, `v1`의 첫 번째 원소를 각각 출력하고 `shape`(또는 값)을 확인합니다.

print(X)

print("==================================")
print(v1)

print("==================================")
print(v1[0])

print("==================================")
print(X.shape)
print(v1.shape)



In [ ]:
# 3. 세 대상이 각각 행렬·벡터·스칼라 중 무엇인지, 분석에서 어떤 역할을 하는지 한 문장씩 설명합니다.

# X는 행렬, v1은 벡터, v1의 첫번쨰 원소는 스칼라

### 문제 1-2 : 벡터의 덧셈과 스칼라 곱 계산하기
1. 두 번째 샘플의 측정값을 `v2`로 추출합니다.
2. 두 샘플의 평균 측정값을 `(v1 + v2) * 0.5`로 계산합니다. (벡터 덧셈 + 스칼라 곱)
3. `v1`의 각 성분을 2배로 키운 `2 * v1`을 계산하고, 원본과 비교합니다.
4. 벡터 덧셈과 스칼라 곱이 각각 어떤 의미를 갖는지 한 문장씩 설명합니다.

In [ ]:
# 1. 두 번째 샘플의 측정값을 `v2`로 추출합니다.
v2 = X.iloc[1].values
print(f"v2: \n {v2}")

# 2. 두 샘플의 평균 측정값을 `(v1 + v2) * 0.5`로 계산합니다. (벡터 덧셈 + 스칼라 곱)
print("==================================")
print(f"v1: \n{v1}")
print(f"v2: \n{v2}")
print("==================================")
print(f"(v1 + v2) * 0.5: \n{(v1 + v2) * 0.5}")

# 3. `v1`의 각 성분을 2배로 키운 `2 * v1`을 계산하고, 원본과 비교합니다.
print("==================================")
print(f"원본: \n{v1}\nv1의 각 성분을 2배로: \n{v1 * 2}")

# 4. 벡터 덧셈과 스칼라 곱이 각각 어떤 의미를 갖는지 한 문장씩 설명합니다.


# 필수 2 : 샘플별 측정값의 "크기"를 하나의 숫자로 요약하기
품질관리팀은 샘플마다 측정 항목이 10개가 넘어 한눈에 비교하기 어렵습니다.
각 샘플의 측정값 묶음이 전체적으로 얼마나 큰지를 숫자 하나로 요약할 수 있으면 이상 샘플을 빠르게 걸러낼 수 있습니다. 노름을 계산해 벡터의 크기를 재고, 크기 차이를 없앤 뒤 방향만 비교할 수 있도록 정규화합니다.

### 문제 2-1 : L1·L2·L∞ 노름 계산하기
1. `v1`에 대해 L1 노름, L2 노름, L∞ 노름을 각각 계산합니다.
2. L2 노름은 `np.sqrt(np.sum(v1 ** 2))`로도 직접 계산해 `np.linalg.norm(v1, 2)`와 값이 같은지 확인합니다.
3. 세 노름이 각각 무엇을 크게 반영하는지 비교해 한 문장으로 설명합니다.

In [ ]:
# 1. `v1`에 대해 L1 노름, L2 노름, L∞ 노름을 각각 계산합니다.
l1_norm = np.linalg.norm(v1, 1)
l2_norm = np.linalg.norm(v1, 2)
linf_norm = np.linalg.norm(v1, np.inf)
print(f"L1 norm is: {l1_norm}, L2 norm is: {l2_norm}, L infinite norm is {linf_norm}")

# 2. L2 노름은 `np.sqrt(np.sum(v1 ** 2))`로도 직접 계산해 `np.linalg.norm(v1, 2)`와 값이 같은지 확인합니다.
l2_norm_passive = np.sqrt(np.sum(v1 ** 2))
if l2_norm == l2_norm_passive:
    print(f"직접 계산한 L2 norm: {l2_norm_passive} vs linalg.norm() 함수로 계산한 값: {l2_norm_passive}")
else:
    print("값이 같지 않음")

# 3. 세 노름이 각각 무엇을 크게 반영하는지 비교해 한 문장으로 설명합니다.
# L1 노름은 각 원소의 절댓값을 더한 값
# L2 노름은 각 원소의 제곱값을 더해서 제곱근을 취해준 값
# L 무한대 노름은 원소들의 절댓값 중 가장 큰 값


### 문제 2-2 : 정규화로 단위벡터 만들기
1. `v1`을 자신의 L2 노름으로 나누어 단위벡터 `u1`을 만듭니다.
2. `u1`의 L2 노름을 계산해 1이 되는지 확인합니다.
3. 앞 5개 샘플을 골라 각각 정규화한 뒤, 정규화 전후 노름을 표로 비교합니다.
4. 정규화가 벡터의 무엇을 유지하고 무엇을 바꾸는지 한 문장으로 설명합니다.

In [ ]:
# 1. `v1`을 자신의 L2 노름으로 나누어 단위벡터 `u1`을 만듭니다.
u1 = v1 / np.linalg.norm(v1)
print("# 1. `v1`을 자신의 L2 노름으로 나누어 단위벡터 `u1`을 만듭니다.")
print(f"계산된 단위벡터: \n{u1}")

# 2. `u1`의 L2 노름을 계산해 1이 되는지 확인합니다.
print("\n\n# 2. `u1`의 L2 노름을 계산해 1이 되는지 확인합니다.")
print(f" 단위벡터의 크기: {np.sqrt(np.sum(u1 ** 2))}")

# 3. 앞 5개 샘플을 골라 각각 정규화한 뒤, 정규화 전후 노름을 표로 비교합니다.
v3 = X.iloc[:5].values
# print(np.linalg.norm(v3, axis=1))
v3_normal = normalize(v3)
print(np.linalg.norm(v3_normal, axis=1))

# 4. 정규화가 벡터의 무엇을 유지하고 무엇을 바꾸는지 한 문장으로 설명합니다.
# 정규화는 벡터를 자기 자신의 노름으로 나누어, 방향은 그대로 유지하고 크기만 1로 통일시킨다.